# Hybrid ALNS Performance Evaluation

---

## I. Purpose

This notebook defines a **controlled benchmark procedure** to evaluate the Hybrid ALNS solver,
which utilizes an offline repair model to guide neighborhood reconstruction.

**Setup** — from the repo root run in order:
```bash
bash migrate.sh      # rename 5_hybrid_ml_metaheuristics → hybrid_ml_metaheuristics
bash migrate_src.sh  # move packages into bin_packing/, editable install
```
Then restart the Jupyter kernel once.

## II. Experimental Preconditions and Reproducibility Controls

Required artifacts:
- trained model `bin_packing/hybrid_ml_metaheuristics/hybrid_alns/models/repair_model_v2.pkl`
- benchmark instance directory (`datasets/` at the repo root)
- fixed random seed and defined iteration / time budgets

In [ ]:
from pathlib import Path

from bin_packing.datasets.registry import DATASET_REGISTRY
from bin_packing.utilities.benchmarking import Benchmark

# Notebook lives at bin_packing/hybrid_ml_metaheuristics/hybrid_alns/
# Three .parents up → repo root
REPO_ROOT  = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path.cwd().parents[2]
MODEL_PATH = REPO_ROOT / "bin_packing" / "hybrid_ml_metaheuristics" / "hybrid_alns" / "models" / "repair_model_v2.pkl"
SOLVER_PATH = REPO_ROOT / "bin_packing" / "hybrid_ml_metaheuristics" / "hybrid_alns" / "solver.py"

assert MODEL_PATH.exists(),  f"Model not found: {MODEL_PATH}"
assert SOLVER_PATH.exists(), f"Solver not found: {SOLVER_PATH}"
print("Model  :", MODEL_PATH)
print("Solver :", SOLVER_PATH)

## III. Hybrid Run (ALNS with Offline Repair Model)

Runs the full benchmark via the `Benchmark` class — no CLI shell-out.

In [ ]:
DATASET = DATASET_REGISTRY["falkenauer-t"]

bench = Benchmark(dataset=DATASET, solver_path=SOLVER_PATH)

bench.run(
    method=None,
    method_args={
        "max_iterations": 2000,
        "model_path": str(MODEL_PATH),
    },
)

results = bench.get_results()
if results:
    csv_path = bench.save_results_to_csv()
    print("Results saved to:", csv_path)

## IV. Summary Statistics

In [ ]:
from bin_packing.utilities.statistics import summarize, summarize_by_size, ResultRow

rows = [
    ResultRow(
        instance_name=r.instance_name,
        dataset_key=r.dataset_key,
        num_items=r.num_items,
        bin_capacity=r.bin_capacity,
        bins_used=r.bins_used,
        lower_bound=r.lower_bound,
        total_weight=r.total_weight,
        elapsed_time=r.elapsed_time,
        method=r.method,
        timed_out=r.timed_out,
    )
    for r in results
]

summary = summarize(rows)
by_size = summarize_by_size(rows)

print("\n=== Overall ===")
for k, v in summary.items():
    print(f"  {k:<30s}: {v:.4f}" if isinstance(v, float) else f"  {k:<30s}: {v}")

print("\n=== By instance size ===")
for s in by_size:
    print(f"  n={s.num_items:4d} | completed={s.completed}/{s.instances}"
          f" | avg_time={s.avg_time_s:.4f}s | avg_gap={s.avg_gap:.3f}")

## V. Graphs

In [ ]:
from bin_packing.utilities.graphing import create_graphs

graph_paths = create_graphs(csv_path)
print("Graphs written:")
for p in graph_paths:
    print(" ", p)